In [46]:
import glob
import os
import re
import h5py
import matplotlib.pyplot as plt
import numpy as np

# 1. SETUP & PATHS CONFIGURATION
root_folder = r"C:\Users\leaga\Downloads\PDCs"
simple_plots_folder = os.path.join(root_folder, "Data Simple Plots")
comparison_plots_folder = os.path.join(root_folder, "Comparison Plots")

# Automatically create both output directories if they don't exist
os.makedirs(simple_plots_folder, exist_ok=True)
os.makedirs(comparison_plots_folder, exist_ok=True)

channels = ["PDC_00", "PDC_01", "PDC_02", "PDC_03"]
channel_colors = ["#0055ff", "#e63946", "#2a9d8f", "#9d4edd"]
colors_palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22","#17becf"]

data_store = {}

# Ignore generated output folders if they live inside root_folder
ignored_folders = {simple_plots_folder, comparison_plots_folder}
subfolders = [
    f.path for f in os.scandir(root_folder)
    if f.is_dir() and f.path not in ignored_folders
]

print(f"--- Loading data & generating individual plots ({len(subfolders)} folders detected) ---")

# 2. SINGLE DATA PASS: INDIVIDUAL PLOTS & STORE MEANS FOR COMPARISONS
for folder_path in subfolders:
    folder_name = os.path.basename(folder_path)
    file_paths = sorted(glob.glob(os.path.join(folder_path, "CTL_048_*.h5")))

    if not file_paths:
        print(f"\n[SKIP] No HDF5 files found in: {folder_name}")
        continue

    print(f"\n-------------------------------------------")
    print(f"Processing folder: {folder_name} ({len(file_paths)} files)")
    print(f"-------------------------------------------")

    data_store[folder_name] = {
        "folder_path": folder_path,
        "raw_name": folder_name,
        "n_files": len(file_paths),
        "means": {},
    }

    # Initialize 2x2 subplot grid for the individual folder
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, (ch, color) in enumerate(zip(channels, channel_colors)):
        data_key = f"TRANSMIT/CTL_048/PDC/{ch}/PDC_DATA/DGTL_SUM/DGTL_SUM"
        curves = []
        bad_files = []

        # Read HDF5 files once
        for path in file_paths:
            with h5py.File(path, "r") as f:
                if data_key not in f:
                    bad_files.append(path)
                    continue
                curves.append(f[data_key][:])

        if bad_files:
            print(f"Channel {ch}: {len(bad_files)} invalid files skipped.")

        if not curves:
            print(f"[ERROR] No valid data found for {ch}, skipping channel.")
            continue

        all_curves = np.array(curves)
        mean_curve = np.mean(all_curves, axis=0)
        std_curve = np.std(all_curves, axis=0)

        # Store mean for later comparisons
        data_store[folder_name]["means"][ch] = mean_curve

        # Plot on corresponding subplot (individual curves + mean + standard deviation band)
        ax = axes[idx]
        ax.plot(all_curves.T, color="gray", alpha=0.08, linewidth=0.5)
        ax.plot(
            mean_curve,
            color=color,
            linewidth=2,
            label=f"Mean ({len(curves)} files)",
        )
        ax.fill_between(
            range(len(mean_curve)),
            mean_curve - std_curve,
            mean_curve + std_curve,
            color=color,
            alpha=0.2,
            label="Standard deviation (±1σ)",
        )

        ax.set_title(f"{ch}", fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(loc="upper right", fontsize=9)
        ax.set_yscale("log")

    fig.supxlabel("Samples (128 points)", fontsize=12)
    fig.supylabel("DGTL_SUM", fontsize=12)
    fig.suptitle(f"Comparative Analysis of 4 PDCs — {folder_name}", fontsize=14)

    plt.tight_layout()

    save_path = os.path.join(
        simple_plots_folder, f"comparative_analysis_4PDCs_{folder_name}.png"
    )
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"--> Individual plot saved: {save_path}")

print(f"\nData loading and individual plots completed for {len(data_store)} conditions.\n")

# 3. COMPARISON FUNCTION
def plot_comparison(condition_dict, title_suffix, filename):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, ch in enumerate(channels):
        ax = axes[idx]

        for color_idx, (label, f_name) in enumerate(condition_dict.items()):
            if f_name in data_store and ch in data_store[f_name]["means"]:
                mean_curve = data_store[f_name]["means"][ch]
                color = colors_palette[color_idx % len(colors_palette)]
                ax.plot(
                    mean_curve,
                    linewidth=1.8,
                    label=label,
                    color=color,
                )

        ax.set_title(f"{ch}", fontsize=11, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(loc="upper right", fontsize=8)
        ax.set_yscale("log")

    fig.supxlabel("Samples", fontsize=12)
    fig.supylabel("DGTL_SUM (Log scale)", fontsize=12)
    fig.suptitle(
        f"PDC Response Comparison — {title_suffix}",
        fontsize=14,
        fontweight="bold",
    )

    plt.tight_layout()

    save_path = os.path.join(comparison_plots_folder, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"--> Comparison saved: {save_path}")


# A. LED Voltage Sweep (Hold-off 2000ns, Pulse Width 100ns)
voltage_series_2000ns = {
    "2.5V": "2000ns_2.5V_100ns",
    "3.0V": "2000ns_3V_100ns",
    "4.0V": "2000ns_4V_100ns",
    "5.0V": "2000ns_5V_100ns",
}
voltage_series_2000ns = {
    k: v for k, v in voltage_series_2000ns.items() if v in data_store
}
if voltage_series_2000ns:
    plot_comparison(
        voltage_series_2000ns,
        "Varying Voltage (Hold-off 2000ns, Width 100ns)",
        "comparison_LED_Voltage_2000ns_100ns.png",
    )

# B. Voltage Sweep at Long Hold-off (25000ns, Pulse Width 2000ns)
voltage_series_25000ns = {
    "2.300V": "25000ns_2.300V_2000ns",
    "2.510V": "25000ns_2.510V_2000ns",
}
voltage_series_25000ns = {
    k: v for k, v in voltage_series_25000ns.items() if v in data_store
}
if voltage_series_25000ns:
    plot_comparison(
        voltage_series_25000ns,
        "Varying Voltage (Hold-off 25000ns, Width 2000ns)",
        "comparison_LED_Voltage_25000ns_2000ns.png",
    )

# C. Hold-off Time Sweep (Fixed LED 5V, Pulse Width 100ns)
holdoff_series_5V = {
    "100ns": "100ns_5V_100ns",
    "300ns": "300ns_5V_100ns",
    "2000ns": "2000ns_5V_100ns",
}
holdoff_series_5V = {
    k: v for k, v in holdoff_series_5V.items() if v in data_store
}
if holdoff_series_5V:
    plot_comparison(
        holdoff_series_5V,
        "Varying Hold-off Time (LED 5V, Width 100ns)",
        "comparison_HoldOff_5V_100ns.png",
    )

# D. Hold-off Time Sweep at 2.510V / 500ns Width
holdoff_series_2510V = {
    "250ns (2.51V)": "250ns_2.510V_500ns",
    "2000ns (2.4V)": "2000ns_2.4V_500ns",
    "2500ns (2.51V)": "2500ns_2.510V_500ns",
}
holdoff_series_2510V = {
    k: v for k, v in holdoff_series_2510V.items() if v in data_store
}
if holdoff_series_2510V:
    plot_comparison(
        holdoff_series_2510V,
        "Varying Hold-off Time (LED ~2.5V, Width 500ns)",
        "comparison_HoldOff_2.510V_500ns.png",
    )

# E. Pulse Width Sweep at 250ns Hold-off (~2.5V)
pulsewidth_series_250ns = {
    "50ns (2.510V)": "250ns_2.510V_50ns",
    "500ns (2.4V)": "250ns_2.4V_500ns",
    "500ns (2.510V)": "250ns_2.510V_500ns",
}
pulsewidth_series_250ns = {
    k: v for k, v in pulsewidth_series_250ns.items() if v in data_store
}
if pulsewidth_series_250ns:
    plot_comparison(
        pulsewidth_series_250ns,
        "Varying Pulse Width (Hold-off 250ns, LED ~2.5V)",
        "comparison_Pulse_Width_150ns_250ns.png",
    )

print("\nProcessing complete! All comparison figures saved in:", comparison_plots_folder)

--- Loading data & generating individual plots (14 folders detected) ---

-------------------------------------------
Processing folder: 100ns_5V_100ns (525 files)
-------------------------------------------
Channel PDC_00: 10 invalid files skipped.
Channel PDC_01: 10 invalid files skipped.
Channel PDC_02: 10 invalid files skipped.
Channel PDC_03: 10 invalid files skipped.
--> Individual plot saved: C:\Users\leaga\Downloads\PDCs\Data Simple Plots\comparative_analysis_4PDCs_100ns_5V_100ns.png

-------------------------------------------
Processing folder: 2000ns_2.4V_500ns (37 files)
-------------------------------------------
--> Individual plot saved: C:\Users\leaga\Downloads\PDCs\Data Simple Plots\comparative_analysis_4PDCs_2000ns_2.4V_500ns.png

-------------------------------------------
Processing folder: 2000ns_2.5V_100ns (434 files)
-------------------------------------------
Channel PDC_00: 12 invalid files skipped.
Channel PDC_01: 12 invalid files skipped.
Channel PDC_02: 12 i